# 通用每日交易快照与视频

本 Notebook 适用于任意包含 `equity_curve_<策略>.csv`、`positions_<策略>.csv` 和 `trades_<策略>.csv` 的回测输出目录。

使用顺序：先运行参数与数据读取单元，再运行预览单元；确认画面后运行并行帧生成单元，最后运行视频合成单元。


In [ ]:
# ==================== 全部可调参数 ====================
from pathlib import Path
import json
import os
import re
import subprocess
import sys

# 项目和数据路径：可以改成任意回测结果目录。
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "backtest_output").is_dir()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "backtest_output" / "5day_top10_long_short"
FRAME_DIR = DATA_DIR / "daily_frames"
VIDEO_PATH = DATA_DIR / "daily_snapshot_video.mp4"

# 资金、画面和视频参数。
INITIAL_CAPITAL = 500_000.0
FPS = 16
FIGURE_SIZE = (18, 12)
FRAME_DPI = 150
MAX_POSITIONS_TO_SHOW = 10
MAX_TRADES_TO_SHOW = 12
PARALLEL_WORKERS = min(4, os.cpu_count() or 1)
FRAME_LIMIT = 0                  # 0 表示生成全部日期；大于 0 可用于测试。
GENERATE_FRAMES = True
BUILD_VIDEO = False              # 确认帧无误后改为 True。

# 策略文件名中的 key 必须对应三个 CSV 文件名中的 <策略> 部分。
# 新增策略时，只需在这里增加一项；布局会自动按策略数量增加行。
STRATEGY_SPECS = [
    {"key": "price_top10_long", "title": "5日涨幅 Top 10 做多", "color": "#16806b"},
    {"key": "price_bottom10_short", "title": "5日跌幅 Bottom 10 做空", "color": "#c34e55"},
]

# 设为 True 时自动读取 DATA_DIR 下全部 equity_curve_*.csv；
# 自动模式标题使用文件名，可再在 STRATEGY_SPECS 中改成更清晰的标题。
AUTO_DISCOVER_STRATEGIES = False

# CSV 编码按顺序尝试；帧文件和视频命名规则也集中在这里。
CSV_ENCODINGS = ("utf-8-sig", "gb18030", "gbk")
FRAME_PATTERN = re.compile(r"^frame_(\d+)_(\d{8})\.png$", re.IGNORECASE)

DATA_DIR = Path(DATA_DIR)
FRAME_DIR = Path(FRAME_DIR)
VIDEO_PATH = Path(VIDEO_PATH)
FRAME_DIR.mkdir(parents=True, exist_ok=True)
print("数据目录：", DATA_DIR.resolve())
print("帧目录：", FRAME_DIR.resolve())
print("视频路径：", VIDEO_PATH.resolve())


In [ ]:
# ==================== 读取并校验回测数据 ====================
import pandas as pd

COLORS = ["#16806b", "#c34e55", "#4768a8", "#9b6ab0", "#d17a22", "#3b8f8f"]

def discover_specs():
    keys = sorted(path.stem.removeprefix("equity_curve_")
                  for path in DATA_DIR.glob("equity_curve_*.csv"))
    return [
        {"key": key, "title": key.replace("_", " ").title(), "color": COLORS[i % len(COLORS)]}
        for i, key in enumerate(keys)
    ]

if AUTO_DISCOVER_STRATEGIES:
    STRATEGY_SPECS = discover_specs()
if not STRATEGY_SPECS:
    raise ValueError("没有配置策略，请填写 STRATEGY_SPECS 或开启 AUTO_DISCOVER_STRATEGIES")

def read_csv(name):
    path = DATA_DIR / name
    for encoding in CSV_ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"无法读取 {path}")

def parse_tables(spec):
    key = spec["key"]
    required = {
        "equity": ["timestamp", "equity"],
        "positions": ["时间", "品种", "合约", "开仓时点", "标记价", "浮动盈亏", "保证金收益率"],
        "trades": ["品种", "开仓时点", "平仓时点", "开仓价", "平仓价", "净盈亏"],
    }
    equity = read_csv(f"equity_curve_{key}.csv")
    positions = read_csv(f"positions_{key}.csv")
    trades = read_csv(f"trades_{key}.csv")
    for name, frame in (("equity", equity), ("positions", positions), ("trades", trades)):
        missing = [column for column in required[name] if column not in frame.columns]
        if missing:
            raise ValueError(f"{key} 的 {name} 文件缺少列：{missing}")
    equity["timestamp"] = pd.to_datetime(equity["timestamp"])
    positions["时间"] = pd.to_datetime(positions["时间"])
    positions["开仓时点"] = pd.to_datetime(positions["开仓时点"])
    for column in ("开仓时点", "平仓时点"):
        trades[column] = pd.to_datetime(trades[column])
    return (equity.sort_values("timestamp"), positions.sort_values("时间"),
            trades.sort_values("开仓时点"))

TABLES = {spec["key"]: parse_tables(spec) for spec in STRATEGY_SPECS}
ALL_DAYS = sorted(set().union(*(set(table[0]["timestamp"].dt.normalize())
                                for table in TABLES.values())))
if not ALL_DAYS:
    raise ValueError("资金曲线中没有可用日期")
ACTIVE_DAYS = [day for day in ALL_DAYS if any(
    ((pos["时间"].dt.normalize() == day).any() or
     (tr["开仓时点"].dt.normalize() == day).any() or
     (tr["平仓时点"].dt.normalize() == day).any())
    for _, pos, tr in TABLES.values()
)]
print(f"策略数：{len(STRATEGY_SPECS)}；全部日期：{len(ALL_DAYS)}；有交易或持仓的日期：{len(ACTIVE_DAYS)}")
print(f"日期范围：{ALL_DAYS[0].date()} 至 {ALL_DAYS[-1].date()}")


In [ ]:
# ==================== 快照绘图函数 ====================
import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

def choose_font():
    for path in (r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\simhei.ttf", r"C:\Windows\Fonts\simsun.ttc"):
        if Path(path).exists():
            return FontProperties(fname=path)
    return FontProperties()

FONT = choose_font()
plt.rcParams["axes.unicode_minus"] = False

def day_positions(positions, day):
    rows = positions[positions["时间"].dt.normalize() <= day]
    if rows.empty:
        return positions.iloc[0:0].copy()
    latest = rows[rows["时间"] == rows["时间"].max()].copy()
    return latest[latest["品种"].fillna("").astype(str).str.strip() != ""].sort_values("浮动盈亏", ascending=False)

def money(value):
    return "—" if pd.isna(value) else f"¥{float(value):,.0f}"

def pct(value):
    return "—" if pd.isna(value) else f"{float(value):+.1%}"

def short_time(value):
    return pd.Timestamp(value).strftime("%H:%M") if pd.notna(value) else "—"

def draw_table(ax, rows, columns, widths, fontsize=7.6, row_height=0.115):
    ax.axis("off")
    if not rows:
        ax.text(0.02, 0.62, "无记录", fontproperties=FONT, fontsize=fontsize + 1, color="#667085")
        return
    table = ax.table(cellText=rows, colLabels=columns, colWidths=widths, loc="upper left", cellLoc="left", bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    for (row, _), cell in table.get_celld().items():
        cell.set_height(row_height)
        cell.set_edgecolor("#d9dee7")
        cell.PAD = 0.025
        if row == 0:
            cell.set_facecolor("#eef2f7")
            cell.set_text_props(weight="bold", color="#263245", fontproperties=FONT)
        else:
            cell.set_facecolor("#ffffff" if row % 2 else "#f8fafc")
            cell.set_text_props(color="#263245", fontproperties=FONT)

def make_snapshot(day, output_path=None, show=True):
    day = pd.Timestamp(day).normalize()
    row_count = len(STRATEGY_SPECS)
    fig = plt.figure(figsize=(FIGURE_SIZE[0], max(FIGURE_SIZE[1], 5.0 * row_count)), facecolor="#ffffff")
    outer = GridSpec(row_count, 1, figure=fig, hspace=0.22)
    fig.suptitle(f"交易日快照  {day:%Y-%m-%d}", fontproperties=FONT, fontsize=21, fontweight="bold", y=0.995)
    all_values = pd.concat([TABLES[s["key"]][0]["equity"] for s in STRATEGY_SPECS])
    global_min, global_max = float(all_values.min()), float(all_values.max())
    for row_no, spec in enumerate(STRATEGY_SPECS):
        key, meta = spec["key"], spec
        equity, positions, trades = TABLES[key]
        sub = GridSpecFromSubplotSpec(2, 2, subplot_spec=outer[row_no], width_ratios=[1.08, 1.55], height_ratios=[0.32, 0.68], wspace=0.12, hspace=0.08)
        ax_curve = fig.add_subplot(sub[:, 0])
        ax_info = fig.add_subplot(sub[0, 1]); ax_info.axis("off")
        ax_table = fig.add_subplot(sub[1, 1])
        day_end = day + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)
        history = equity[equity["timestamp"] <= day_end]
        if history.empty:
            history = equity.iloc[:1]
        current = history.iloc[-1]
        ax_curve.plot(history["timestamp"], history["equity"], color=meta.get("color", "#16806b"), linewidth=2.0)
        ax_curve.scatter([current["timestamp"]], [current["equity"]], color=meta.get("color", "#16806b"), s=44, zorder=4)
        ax_curve.axvline(current["timestamp"], color=meta.get("color", "#16806b"), alpha=0.22, linewidth=1.1)
        x_pad = pd.Timedelta(days=18)
        y_pad = max((global_max - global_min) * 0.07, INITIAL_CAPITAL * 0.03)
        ax_curve.set_xlim(equity["timestamp"].min() - x_pad, equity["timestamp"].max() + x_pad)
        ax_curve.set_ylim(global_min - y_pad, global_max + y_pad)
        ax_curve.axhline(INITIAL_CAPITAL, color="#9aa4b2", linestyle="--", linewidth=1)
        ax_curve.set_title(meta.get("title", key), fontproperties=FONT, fontsize=14, loc="left", color=meta.get("color", "#16806b"), pad=8)
        ax_curve.set_ylabel("组合权益（元）", fontproperties=FONT)
        ax_curve.grid(axis="y", color="#e7ebf0", linewidth=0.8)
        ax_curve.spines[["top", "right"]].set_visible(False)
        ax_curve.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=6))
        ax_curve.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax_curve.tick_params(axis="x", rotation=20)
        for label in ax_curve.get_xticklabels() + ax_curve.get_yticklabels():
            label.set_fontproperties(FONT)
        current_positions = day_positions(positions, day)
        opened = trades[trades["开仓时点"].dt.normalize() == day]
        closed = trades[trades["平仓时点"].dt.normalize() == day]
        realized = float(closed["净盈亏"].sum()) if not closed.empty else 0.0
        info = [f"期末权益  {money(current['equity'])}", f"累计收益  {money(float(current['equity']) - INITIAL_CAPITAL)} ({pct(float(current['equity']) / INITIAL_CAPITAL - 1)})", f"当日已实现盈亏  {money(realized)}", f"当前持仓  {len(current_positions)}    当日开仓 {len(opened)}    当日平仓 {len(closed)}"]
        ax_info.text(0, 0.92, "\n".join(info), va="top", fontproperties=FONT, fontsize=11, linespacing=1.45, color="#263245")
        combined = []
        for _, item in current_positions.head(MAX_POSITIONS_TO_SHOW).iterrows():
            combined.append(["持仓", str(item.get("品种", "")), str(item.get("合约", "")), short_time(item.get("开仓时点")), money(item.get("标记价")), money(item.get("浮动盈亏")), pct(item.get("保证金收益率"))])
        for _, trade in pd.concat([opened, closed]).head(MAX_TRADES_TO_SHOW).iterrows():
            is_closed = pd.notna(trade.get("平仓时点")) and trade["平仓时点"].normalize() == day
            combined.append(["平仓" if is_closed else "开仓", str(trade.get("品种", "")), "", short_time(trade.get("平仓时点" if is_closed else "开仓时点")), money(trade.get("平仓价" if is_closed else "开仓价")), money(trade.get("净盈亏")) if is_closed else "—", "—"])
        columns = ["状态", "品种/动作", "合约/品种", "时间", "价格", "浮盈/已实现", "保证金收益率"]
        if combined:
            draw_table(ax_table, combined, columns, [0.10, 0.20, 0.16, 0.13, 0.15, 0.18, 0.14])
        else:
            ax_table.axis("off"); ax_table.text(0, 0.7, "当前无持仓，今日无交易", fontproperties=FONT, fontsize=10, color="#667085")
    fig.text(0.01, 0.01, "资金曲线、持仓与交易均来自回测输出 CSV；平仓计入已实现盈亏，未平仓显示浮动盈亏。", fontproperties=FONT, fontsize=9, color="#667085")
    if output_path is not None:
        fig.savefig(output_path, dpi=FRAME_DPI, bbox_inches="tight", facecolor="white")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return output_path


In [ ]:
# ==================== 预览一帧 ====================
PREVIEW_DATE = ACTIVE_DAYS[0] if ACTIVE_DAYS else ALL_DAYS[0]
PREVIEW_PATH = FRAME_DIR / f"preview_{PREVIEW_DATE:%Y%m%d}.png"
make_snapshot(PREVIEW_DATE, PREVIEW_PATH, show=True)
print("预览文件：", PREVIEW_PATH.resolve())


确认预览无误后，运行下面的并行生成单元。`GENERATE_FRAMES` 为 False 时不会生成全部帧。

In [ ]:
# ==================== 并行生成每日帧 ====================
if GENERATE_FRAMES:
    generator = PROJECT_ROOT / "scripts" / "generate_daily_frames.py"
    command = [
        sys.executable, str(generator),
        "--data-dir", str(DATA_DIR),
        "--frame-dir", str(FRAME_DIR),
        "--workers", str(PARALLEL_WORKERS),
        "--initial-capital", str(INITIAL_CAPITAL),
        "--strategy-specs", json.dumps(STRATEGY_SPECS, ensure_ascii=False),
    ]
    if FRAME_LIMIT:
        command.extend(["--limit", str(FRAME_LIMIT)])
    subprocess.run(command, check=True)
else:
    print("已跳过批量帧生成。")


In [ ]:
# ==================== 读取每日帧并合成 MP4 ====================
if BUILD_VIDEO:
    import tempfile
    import imageio_ffmpeg

    frames = [path for path in FRAME_DIR.glob("frame_*.png") if FRAME_PATTERN.match(path.name)]
    frames.sort(key=lambda path: (int(FRAME_PATTERN.match(path.name).group(1)), FRAME_PATTERN.match(path.name).group(2)))
    if not frames:
        raise FileNotFoundError(f"{FRAME_DIR.resolve()} 中没有找到每日 PNG 帧")
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as handle:
        concat_file = Path(handle.name)
        for frame in frames:
            filename = frame.resolve().as_posix().replace("'", "'\\''")
            handle.write(f"file '{filename}'\n")
    try:
        command = [
            imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-f", "concat", "-safe", "0",
            "-i", str(concat_file), "-r", str(FPS),
            "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
            "-c:v", "libx264", "-pix_fmt", "yuv420p", str(VIDEO_PATH),
        ]
        subprocess.run(command, check=True)
    finally:
        concat_file.unlink(missing_ok=True)
    print(f"视频已生成：{VIDEO_PATH.resolve()}")
    print(f"帧数：{len(frames)}；帧率：{FPS} FPS；时长约：{len(frames) / FPS:.1f} 秒")
else:
    print("已跳过视频合成；确认帧无误后将 BUILD_VIDEO 改为 True。")
